In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from collections import Counter
from pathlib import Path
import json

BASE = "https://115.159.159.82"
api_url = BASE + "/api/devices"

r = requests.get(api_url, params={"page": 1, "page_size": 50}, timeout=10)
print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))


status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json


In [2]:
data = r.json()
print(type(data), data.keys())
print(data)


<class 'dict'> dict_keys(['page', 'page_size', 'total', 'items'])
{'page': 1, 'page_size': 50, 'total': 5000, 'items': [{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}, {'device_id': 'D00000002', 'company_id': 'C000001', 'device_name': '循环泵-000001-002', 'device_type': 'pump', 'model': 'SP-PUM-500', 'zone': '智造一区', 'install_date': '2016-03-06', 'rated_power_kw': 73.8, 'sampling_interval_s': 60, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-08T05:44:00'}, {'device_id': 'D00000003', 'company_id': 'C000001', 'device_name': '空压机-000001-003', 'device_type': 'compressor', 'model': 'SP-COM-200', 'zone': '智造一区', 'install_date': '2016-12-18', 'rated_power_kw': 248.6, 'sampling_interval_s': 30, 'protocol': 'modbus', 'status':

In [3]:
print(data["items"][0])


{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}


In [4]:
total = data["total"]
page_size = data["page_size"]
total_pages = (total + page_size - 1) // page_size

print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)


total: 5000
page_size: 50
total_pages: 100


In [6]:
devices = []

for page in range(1, total_pages + 1):
    r = requests.get(
        api_url,
        params={"page": page, "page_size": page_size},
        timeout=10
    )
    page_data = r.json()
    devices.extend(page_data["items"])

print("device count:", len(devices))

device count: 5000


In [7]:
base_url = BASE
current_url = base_url + "/companies"

r = requests.get(current_url, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

cards = soup.select("article.company-card")
print("本页企业数：", len(cards))

本页企业数： 20


In [8]:
card = cards[0]

company_id = card.get("data-company-id")
name = card.select_one("h2 a").get_text(strip=True)
status = card.select_one("span.badge").get_text(strip=True)

details = {}
for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

industry = details["所属行业"]
zone = details["所在区域"]
device_count = int(details["设备数量"].replace("台", "").strip())

print(company_id, name, status, industry, zone, device_count)

C000001 临港精工001有限公司 运营中 汽车零部件 智造一区 6


In [9]:
companies = []
for card in cards:
    details = {}
    for item in card.select("dl > div"):
        details[item.select_one("dt").get_text(strip=True)] = item.select_one("dd").get_text(strip=True)
    companies.append({
        "company_id": card.get("data-company-id"),
        "name": card.select_one("h2 a").get_text(strip=True),
        "industry": details["所属行业"],
        "zone": details["所在区域"],
        "device_count": int(details["设备数量"].replace("台", "").strip()),
        "status": card.select_one("span.badge").get_text(strip=True),
    })

print("本页企业数：", len(companies))

本页企业数： 20


In [10]:
next_link = None
for a in soup.select("nav.pagination a"):
    if "下一页" in a.get_text(" ", strip=True):
        next_link = a
        break

print(next_link.get_text(" ", strip=True))
print(next_link.get("href"))

下一页
/companies?page=2


In [11]:
companies = []
current_url = base_url + "/companies"

for i in range(3):
    resp = requests.get(current_url, timeout=10)
    page_soup = BeautifulSoup(resp.text, "html.parser")
    cards = page_soup.select("article.company-card")

    for card in cards:
        details = {}
        for item in card.select("dl > div"):
            details[item.select_one("dt").get_text(strip=True)] = item.select_one("dd").get_text(strip=True)
        companies.append({
            "company_id": card.get("data-company-id"),
            "name": card.select_one("h2 a").get_text(strip=True),
            "industry": details["所属行业"],
            "zone": details["所在区域"],
            "device_count": int(details["设备数量"].replace("台", "").strip()),
            "status": card.select_one("span.badge").get_text(strip=True),
        })
    print(f"第 {i+1} 页：{len(cards)} 条")

    next_link = None
    for a in page_soup.select("nav.pagination a"):
        if "下一页" in a.get_text(" ", strip=True):
            next_link = a
            break
    if next_link is None or i == 2:
        break
    current_url = urljoin(current_url, next_link.get("href"))

print("Company 总数：", len(companies))

第 1 页：20 条
第 2 页：20 条
第 3 页：20 条
Company 总数： 60


In [12]:
print("Company 总数:", len(companies))
print("唯一 company_id 数:", len(set(x["company_id"] for x in companies)))
print("Device 总数:", len(devices))
print("唯一 device_id 数:", len(set(x["device_id"] for x in devices)))

device_counts = Counter(x["company_id"] for x in devices)
mismatch_count = 0
for company in companies:
    if company["device_count"] != device_counts.get(company["company_id"], 0):
        mismatch_count += 1
print("Company-Device 不一致企业数:", mismatch_count)

Company 总数: 60
唯一 company_id 数: 60
Device 总数: 5000
唯一 device_id 数: 5000
Company-Device 不一致企业数: 0


In [13]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("数据已保存到 data/ 目录")

数据已保存到 data/ 目录
